# Real o IA · Entrenamiento en Google Colab
CNN binaria desde cero y **k-means independiente** con características manuales.

REAL=0, FAKE=1. El test oficial se reserva para una única evaluación final. Las etiquetas no intervienen en el ajuste de k-means; se usan después para asociar los grupos con clases.


In [ ]:
# Ejecutar en una sesión limpia de Colab, antes de importar TensorFlow.
# TensorFlow 2.20 admite Python 3.9–3.13; tf-keras conserva el formato Keras 2.
%pip install -q "tensorflow==2.20.0" "tf-keras==2.20.1" "scikit-learn>=1.5,<2" pillow matplotlib seaborn

# Después de instalar, reinicia la sesión desde el menú Entorno de ejecución
# y continúa en la siguiente celda. No instales tensorflowjs: no se necesita.


In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import json, random, zipfile, hashlib, platform
from pathlib import Path
import numpy as np
import tensorflow as tf
import tf_keras as keras
import sklearn
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, adjusted_rand_score, silhouette_score)
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__, 'tf-keras:', keras.__version__, 'scikit-learn:', sklearn.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


## 1. Cargar CIFAKE
Sube el ZIP descargado de Kaggle. Si ya lo tienes en Drive, establece `ZIP_PATH` con su ruta y omite la subida. No subas tu dataset al repositorio del frontend.


In [ ]:
ZIP_PATH = ''  # Ejemplo: /content/drive/MyDrive/cifake.zip
if not ZIP_PATH:
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Sube exactamente un ZIP de CIFAKE.')
    ZIP_PATH = archives[0]
    del uploaded
DATA_ROOT = Path('/content/cifake')
DATA_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as archive:
    for item in archive.infolist():
        target = (DATA_ROOT / item.filename).resolve()
        if not target.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError('El ZIP contiene una ruta fuera de la carpeta de datos.')
    archive.extractall(DATA_ROOT)
train_dirs = [p for p in DATA_ROOT.rglob('train') if (p/'REAL').is_dir() and (p/'FAKE').is_dir()]
if len(train_dirs) != 1:
    raise ValueError('No se encontró una única estructura train/REAL y train/FAKE.')
ROOT = train_dirs[0].parent

def discover(split):
    paths, labels = [], []
    for label, name in enumerate(['REAL', 'FAKE']):
        found = sorted(p for p in (ROOT/split/name).iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'})
        expected = 50000 if split == 'train' else 10000
        if len(found) != expected:
            raise ValueError(f'{split}/{name}: se esperaban {expected} imágenes, hay {len(found)}.')
        paths.extend(map(str, found)); labels.extend([label]*len(found))
    return np.array(paths), np.array(labels, dtype=np.int32)
all_train_paths, all_train_y = discover('train')
test_paths, test_y = discover('test')
print('Archivos oficiales train / test:', len(all_train_paths), len(test_paths))


## 2. Preprocesamiento y particiones sin duplicados exactos
Se usan imágenes RGB de 32 × 32, valores en [0,1]. Para archivos de otro tamaño, el frontend muestrea el centro de cada celda con vecino más próximo, sin recorte. Los modelos reciben los mismos píxeles.

**Protocolo de limpieza:** antes de dividir train/validación, conservar un representante por hash SHA-256 de archivo en el train oficial. Excluir del conjunto de entrenamiento las imágenes también presentes en test, sin consultar sus etiquetas. Después dividir el conjunto limpio de forma estratificada (80/20). No borrar archivos del disco ni modificar el test oficial. Una imagen idéntica con etiquetas de entrenamiento contradictorias exige revisión.

Se imprimen y exportan los recuentos reales y el registro `split_audit`. Los tamaños de entrenamiento serán menores si hay duplicados. Los hashes de bytes no detectan imágenes visualmente idénticas recodificadas ni casi duplicados. Los duplicados internos de test se informan; al conservar el test oficial, su métrica sigue ponderando esas copias.


In [ ]:
def digest(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def clean_training_records(paths, labels, held_out_paths, hash_file=digest):
    """Un representante por archivo idéntico; test oficial intacto.

    Se excluye de train cualquier archivo cuyo hash figure en test,
    sin consultar las etiquetas de test. No se borran archivos del disco.
    El hash detecta duplicados byte a byte, no similitud perceptual.
    """
    test_digests = [hash_file(path) for path in held_out_paths]
    test_hashes = set(test_digests)
    representatives = {}
    duplicates = 0
    for path, label in zip(paths, labels):
        key = hash_file(path)
        label = int(label)
        if key in representatives:
            if representatives[key][1] != label:
                raise ValueError('Una imagen idéntica tiene etiquetas REAL y FAKE en train. Revisar el dataset antes de continuar.')
            duplicates += 1
        else:
            representatives[key] = (path, label)
    overlap = sum(key in test_hashes for key in representatives)
    retained = [record for key, record in representatives.items() if key not in test_hashes]
    if len({label for _, label in retained}) != 2:
        raise ValueError('La limpieza no dejó suficientes clases para entrenar.')
    report = {'hash':'sha256_file_bytes', 'official_train_size':len(paths),
              'removed_duplicate_train_copies':duplicates,
              'excluded_unique_train_images_present_in_test':overlap,
              'clean_train_pool_size':len(retained),
              'official_test_size':len(held_out_paths),
              'duplicate_copies_within_test':len(test_digests)-len(test_hashes),
              'test_policy':'official test unchanged; no test labels used for cleaning'}
    return [path for path, _ in retained], [label for _, label in retained], report

clean_paths, clean_y, split_audit = clean_training_records(all_train_paths, all_train_y, test_paths)
train_paths, val_paths, train_y, val_y = train_test_split(
    np.array(clean_paths), np.array(clean_y, dtype=np.int32),
    test_size=0.2, stratify=clean_y, random_state=SEED)
train_hashes = set(map(digest, train_paths))
val_hashes = set(map(digest, val_paths))
test_hashes = set(map(digest, test_paths))
assert not train_hashes & val_hashes, 'Persisten duplicados train-validación.'
assert not (train_hashes | val_hashes) & test_hashes, 'Persisten cruces con test.'
print(json.dumps(split_audit, indent=2, ensure_ascii=False))
print('Train / validación / test:', len(train_paths), len(val_paths), len(test_paths))
for name, labels in [('train',train_y), ('validación',val_y), ('test',test_y)]:
    print(name, dict(zip(['REAL','FAKE'], np.bincount(labels, minlength=2).tolist())))

def read_rgb(path):
    with Image.open(path) as image:
        arr = np.asarray(image.convert('RGB'), dtype=np.uint8)
    h, w = arr.shape[:2]
    ys = np.minimum(((np.arange(32)+0.5)*h/32).astype(int), h-1)
    xs = np.minimum(((np.arange(32)+0.5)*w/32).astype(int), w-1)
    return arr[ys[:,None], xs[None,:]].astype(np.float32)/255.0

def dataset(paths, labels, training=False):
    def generate():
        for path, label in zip(paths, labels):
            yield read_rgb(path), np.float32(label)
    ds = tf.data.Dataset.from_generator(generate, output_signature=(
        tf.TensorSpec((32,32,3), tf.float32), tf.TensorSpec((), tf.float32)))
    if training:
        ds = ds.shuffle(10000, seed=SEED)
    return ds.batch(128).prefetch(tf.data.AUTOTUNE)
train_ds = dataset(train_paths, train_y, True)
val_ds = dataset(val_paths, val_y)
test_ds = dataset(test_paths, test_y)


## 3. CNN desde cero
Inicialización aleatoria, sin transferencia de aprendizaje. Early stopping y selección de pesos usan exclusivamente la validación. El umbral queda fijado en 0.5; el puntaje sigmoide no garantiza una probabilidad calibrada.


In [ ]:
model = keras.Sequential([
    keras.layers.InputLayer(input_shape=(32,32,3)),
    keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.4),
    keras.layers.Dense(1, activation='sigmoid'),
], name='cifake_cnn_from_scratch')
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
history = model.fit(train_ds, validation_data=val_ds, epochs=40, callbacks=[
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
])
fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, metric in zip(axes, ['loss','accuracy']):
    ax.plot(history.history[metric], label='Entrenamiento')
    ax.plot(history.history['val_'+metric], label='Validación')
    ax.set(xlabel='Época', ylabel=metric); ax.legend()
plt.tight_layout(); plt.show()


## 4. Características independientes y k-means
35 características manuales, sin usar activaciones ni pesos de la CNN:
- 3 medias y 3 desviaciones estándar RGB (poblacionales).
- Histogramas RGB de 8 intervalos por canal, normalizados por 1024.
- Media y desviación de diferencias absolutas horizontales y verticales en gris.
- Varianza del laplaciano interior de 4 vecinos.

Gris = 0.299R + 0.587G + 0.114B. El orden y las fórmulas deben mantenerse iguales en TypeScript. StandardScaler se ajusta solo con train. No hay PCA ni características preentrenadas.


In [ ]:
FEATURE_VERSION = 'rgb-hist-gradient-laplacian-v1'
def features(rgb):
    a = np.asarray(rgb, dtype=np.float64)
    result = list(a.mean(axis=(0,1))) + list(a.std(axis=(0,1)))
    for c in range(3):
        bins = np.minimum((a[:,:,c]*8).astype(int), 7)
        result.extend(np.bincount(bins.ravel(), minlength=8)/1024)
    gray = a[:,:,0]*0.299 + a[:,:,1]*0.587 + a[:,:,2]*0.114
    dx = np.abs(np.diff(gray, axis=1)); dy = np.abs(np.diff(gray, axis=0))
    result.extend([dx.mean(), dx.std(), dy.mean(), dy.std()])
    lap = gray[:-2,1:-1]+gray[2:,1:-1]+gray[1:-1,:-2]+gray[1:-1,2:]-4*gray[1:-1,1:-1]
    result.append(lap.var())
    return np.array(result, dtype=np.float64)

def extract(paths):
    return np.stack([features(read_rgb(p)) for p in paths])
X_train = extract(train_paths)
X_val = extract(val_paths)
scaler = StandardScaler().fit(X_train)
Z_train = scaler.transform(X_train)
kmeans = KMeans(n_clusters=2, random_state=SEED, n_init=10, max_iter=300)
train_clusters = kmeans.fit_predict(Z_train)  # Sin etiquetas.
# Correspondencia uno-a-uno que maximiza aciertos en entrenamiento.
identity = np.mean(train_clusters == train_y)
reverse = np.mean((1-train_clusters) == train_y)
cluster_to_class = np.array([0,1] if identity >= reverse else [1,0], dtype=np.int32)
print('Grupo → clase:', dict(enumerate(cluster_to_class.tolist())))
print('Contingencia train (filas: REAL/FAKE; columnas: grupos):')
print(confusion_matrix(train_y, train_clusters))
print('Accuracy asociada validación:', accuracy_score(val_y, cluster_to_class[kmeans.predict(scaler.transform(X_val))]))


## 5. Evaluación final en test
No se modifican hiperparámetros ni correspondencias a partir de estos resultados. La accuracy de k-means usa la asociación fijada con entrenamiento; ARI evalúa agrupación sin depender del nombre del grupo. Silhouette se estima con una muestra de hasta 2000 ejemplos para limitar memoria. No es comparable directamente con accuracy.


In [ ]:
def classification_metrics(y, pred):
    return {'accuracy':float(accuracy_score(y,pred)),
            'precision':float(precision_score(y,pred,zero_division=0)),
            'recall':float(recall_score(y,pred,zero_division=0)),
            'f1':float(f1_score(y,pred,zero_division=0)),
            'confusion_matrix':confusion_matrix(y,pred,labels=[0,1]).tolist()}
cnn_scores = model.predict(test_ds, verbose=1).ravel()
cnn_pred = (cnn_scores >= 0.5).astype(int)
cnn_metrics = classification_metrics(test_y, cnn_pred)
cnn_metrics['roc_auc'] = float(roc_auc_score(test_y,cnn_scores))
X_test = extract(test_paths)
Z_test = scaler.transform(X_test)
test_clusters = kmeans.predict(Z_test)
km_pred = cluster_to_class[test_clusters]
km_metrics = classification_metrics(test_y,km_pred)
km_metrics['ari'] = float(adjusted_rand_score(test_y,test_clusters))
km_metrics['silhouette'] = (float(silhouette_score(Z_test,test_clusters,sample_size=min(2000,len(Z_test)),random_state=SEED))
                            if len(np.unique(test_clusters)) == 2 else None)
print('CNN:', json.dumps(cnn_metrics,indent=2))
print('K-means:', json.dumps(km_metrics,indent=2))
fig, axes = plt.subplots(1,2,figsize=(10,4))
for ax, name, metrics in zip(axes,['CNN','K-means'],[cnn_metrics,km_metrics]):
    sns.heatmap(metrics['confusion_matrix'],annot=True,fmt='d',cbar=False,ax=ax,
                xticklabels=['REAL','FAKE'],yticklabels=['REAL','FAKE'])
    ax.set(title=name,xlabel='Predicción',ylabel='Clase real')
plt.tight_layout(); plt.show()


## 6. Exportar para la aplicación
Descarga `real-o-ia-models.zip` y extrae su contenido dentro de `public/models/` en el proyecto. El ZIP incluye la CNN en TensorFlow.js, los centroides y escalador de k-means, métricas y un caso de paridad numérica.

La aplicación no inventa resultados si estos archivos no están instalados. JPEG puede decodificarse con pequeñas diferencias entre Pillow y el navegador; el caso de paridad verifica las fórmulas con los mismos píxeles, no identidad entre decodificadores.

La exportación de esta arquitectura Sequential se escribe directamente como `model.json` + shards float32. No requiere instalar el conversor Python `tensorflowjs`, JAX ni TensorFlow Decision Forests. La validación de carga e inferencia en navegador debe compararse con `parity.json` al disponer de los pesos entrenados.


In [ ]:
def export_tfjs_layers(model, destination):
    """Exportar esta CNN tf-keras en formato TF.js Layers, sin conversor externo.

    Solo arquitectura Sequential con capas estándar y pesos float32.
    Especificación: tensorflow/tfjs, write_weights.py y keras_h5_conversion.py.
    No exporta estado del optimizador ni operaciones de entrenamiento.
    """
    allowed = {'InputLayer', 'Conv2D', 'MaxPooling2D', 'Dropout',
               'GlobalAveragePooling2D', 'Dense'}
    topology = json.loads(model.to_json())
    if topology['class_name'] != 'Sequential':
        raise ValueError('El exportador requiere la CNN Sequential de este notebook.')
    for layer in topology['config']['layers']:
        if layer['class_name'] not in allowed:
            raise ValueError(f"Capa no admitida: {layer['class_name']}")
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    specs, chunks, names = [], [], set()
    for weight in model.weights:
        array = np.asarray(weight.numpy())
        if array.dtype != np.float32 or not np.isfinite(array).all():
            raise ValueError(f'Peso inválido o distinto de float32: {weight.name}')
        name = weight.name.removesuffix(':0')
        if name in names:
            raise ValueError(f'Nombre de peso duplicado: {name}')
        names.add(name)
        specs.append({'name':name, 'shape':list(array.shape), 'dtype':'float32'})
        chunks.append(array.astype('<f4', copy=False).tobytes(order='C'))
    payload = b''.join(chunks)
    shard_size = 4 * 1024 * 1024
    count = max(1, (len(payload)+shard_size-1)//shard_size)
    paths = []
    for i in range(count):
        filename = f'group1-shard{i+1}of{count}.bin'
        (destination/filename).write_bytes(payload[i*shard_size:(i+1)*shard_size])
        paths.append(filename)
    artifact = {'format':'layers-model', 'generatedBy':f'tf-keras {keras.__version__}',
                'convertedBy':'real-o-ia Sequential float32 exporter v1',
                'modelTopology':topology,
                'weightsManifest':[{'paths':paths, 'weights':specs}]}
    (destination/'model.json').write_text(json.dumps(artifact, allow_nan=False))
    # Verificar que el manifiesto describe exactamente los bytes escritos.
    expected_bytes = sum(int(np.prod(w['shape']))*4 for w in specs)
    assert expected_bytes == sum((destination/name).stat().st_size for name in paths)

OUT = Path('/content/exported_models')
OUT.mkdir(exist_ok=True)
export_tfjs_layers(model, OUT/'cnn')
model.save('/content/cnn.keras')
km_export = {'schema_version':1,'feature_version':FEATURE_VERSION,'input_size':32,
    'mean':scaler.mean_.tolist(),'scale':scaler.scale_.tolist(),
    'centers':kmeans.cluster_centers_.tolist(),'cluster_to_class':cluster_to_class.tolist()}
(OUT/'kmeans.json').write_text(json.dumps(km_export,allow_nan=False))
metrics_export = {'schema_version':1,'dataset':'CIFAKE','labels':['REAL','FAKE'],
    'train_size':len(train_paths),'validation_size':len(val_paths),'test_size':len(test_paths),
    'seed':SEED,'split_audit':split_audit,'cnn':cnn_metrics,'kmeans':km_metrics,
    'versions':{'tensorflow':tf.__version__,'sklearn':sklearn.__version__,'python':platform.python_version()},
    'history':{k:[float(v) for v in values] for k,values in history.history.items()}}
(OUT/'metrics.json').write_text(json.dumps(metrics_export,allow_nan=False))
# Una imagen determinista, ajena al dataset, para comprobar paridad Python/JS.
parity_rgb = np.random.default_rng(SEED).integers(0,256,size=(32,32,3),dtype=np.uint8)
parity_input = parity_rgb.astype(np.float32)/255
parity_features = features(parity_input)
parity_z = scaler.transform([parity_features])[0]
parity_distances = np.linalg.norm(kmeans.cluster_centers_-parity_z,axis=1)
(OUT/'parity.json').write_text(json.dumps({'rgb':parity_rgb.ravel().tolist(),
    'features':parity_features.tolist(),'distances':parity_distances.tolist(),
    'cnn_score':float(model.predict(parity_input[None],verbose=0)[0,0])},allow_nan=False))
with zipfile.ZipFile('/content/real-o-ia-models.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for path in OUT.rglob('*'):
        if path.is_file(): archive.write(path,path.relative_to(OUT))
files.download('/content/real-o-ia-models.zip')


## Interpretación y referencias
Un grupo llamado REAL puede contener imágenes sintéticas. El indicador de acuerdo solo informa si las dos predicciones coinciden; **no verifica autenticidad**. El puntaje de la CNN no está calibrado y las distancias de k-means no son probabilidades. Registrar limitaciones, errores y resultados tal como se obtengan, sin prometer una accuracy mínima.

CIFAKE usa imágenes pequeñas y un generador concreto. Probar imágenes actuales o de alta resolución supone un cambio de distribución: esta evaluación no valida detección general de ChatGPT, Gemini u otros generadores. La reducción a 32 × 32 elimina información.

Citar al entregar el proyecto:
- Krizhevsky, A., & Hinton, G. (2009). *Learning multiple layers of features from tiny images*.
- Bird, J. J., & Lotfi, A. (2024). *CIFAKE: Image Classification and Explainable Identification of AI-Generated Synthetic Images*. IEEE Access.
- Dataset: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images

Conservar los avisos de licencia del dataset al redistribuirlo. Este repositorio no incluye imágenes de CIFAKE.
